# Lezione 1 — Dati mancanti: capire il problema, poi risolverlo

**Come si usa questo notebook.** Leggi dall'alto verso il basso ed esegui ogni
cella di codice (Shift+Invio). A metà trovi un esercizio: prova prima da solo
nella cella indicata, poi confronta con la soluzione spiegata subito sotto.
Nell'ultima sezione applichi quello che hai imparato al **progetto del corso**
(il "Memory AI Lab"), che crescerà di un pezzo a ogni lezione — oggi ne
costruiamo il primo componente.

**Cosa saprai fare alla fine:** davanti a *qualsiasi* tabella con dei buchi —
un sondaggio, un log applicativo, dati clinici, letture di sensori — saprai
decidere in modo motivato cosa scartare, cosa riempire e come non nascondere
le tue decisioni.

## Parte 1 — Il problema: perché i dati mancanti sono ovunque

Quasi ogni dataset reale ha dei buchi, e le cause sono sempre riconducibili a
un piccolo insieme di motivi, che si ripetono identici in domini diversi:

| Dominio | Perché mancano dati |
|---|---|
| Sondaggi | le persone saltano le domande scomode |
| Medicina | i pazienti abbandonano lo studio |
| Sensori / IoT | la rete perde pacchetti, il sensore si guasta |
| Log applicativi | un campo è stato aggiunto dopo, i vecchi record non ce l'hanno |
| E-commerce | l'utente non compila il campo opzionale |

L'errore da principiante è trattare il buco come "rumore da far sparire" —
cancellarlo o riempirlo alla svelta per avere una tabella pulita da guardare.
Il punto di questa lezione è l'opposto: **il buco è un'informazione**. Il modo
in cui manca ti dice qualcosa sul processo che ha generato i dati, e quella
informazione va usata per decidere cosa fare, non buttata via insieme al
buco stesso.

## Parte 2 — Teoria: i tre motivi per cui un dato manca

Nel 1976 lo statistico Donald Rubin ha proposto una classificazione che oggi
è lo standard per ragionare sui dati mancanti. Non è un dettaglio storico:
è lo strumento che userai ogni volta che vedi una cella vuota. Ci sono tre
casi, e li spieghiamo con lo **stesso esempio concreto** — un termometro che
registra la temperatura ogni ora — visto da tre angolazioni diverse.

**Caso 1 — MCAR (*Missing Completely At Random*).**
Una volta al mese, a un orario a caso, il termometro perde la connessione
Wi-Fi per un secondo e quella lettura va persa. Non ha alcuna relazione con
la temperatura di quel momento, né con nient'altro nella tabella: è
sfortuna pura. Conseguenza pratica: le righe che *hai* sono ancora un
campione rappresentativo di tutte le ore. Puoi scartare le righe con buchi,
o imputarle, senza distorcere le tue conclusioni.

**Caso 2 — MAR (*Missing At Random*).**
Il termometro è alimentato a batteria, e la batteria dura meno quando fa
molto freddo (il litio perde efficienza a basse temperature). Quindi le
letture mancano più spesso in inverno. Qui il buco *dipende da qualcosa*,
ma quel qualcosa — la stagione, che è nella tua tabella come colonna data —
lo puoi osservare. Conseguenza pratica: puoi ancora correggere la
distorsione, perché conosci la variabile che la causa (ad esempio imputando
separatamente per stagione, o includendo la stagione come informazione).

**Caso 3 — MNAR (*Missing Not At Random*).**
Il termometro va in errore quando la temperatura supera i 40°C — il sensore
stesso non regge le temperature estreme e smette di rispondere. Qui il buco
dipende **dal valore che manca**, non da qualcosa che vedi altrove. Questo è
il caso pericoloso: le righe che hai in mano non sono più rappresentative,
perché mancano sistematicamente proprio le giornate più calde. Se scarti
quelle righe, o le riempi con una media calcolata sulle giornate rimaste,
stai sottostimando sistematicamente le temperature estreme — e **nessuna
tecnica automatica di pulizia dati può accorgersene o correggerlo da sola**.
Serve che tu, guardando il processo che ha generato i dati, riconosca il
rischio.

**Il punto pratico che porti a casa:** da una tabella piccola, guardando
solo i numeri, *non puoi dimostrare* con certezza in quale dei tre casi ti
trovi — servirebbe conoscere il processo reale che ha generato i dati (come
nell'esempio del sensore che si guasta col caldo). Per questo la regola
professionale non è "applica sempre la stessa tecnica", ma: **fai
un'assunzione esplicita sul meccanismo, scrivila da qualche parte, e rendi
tracciabile ogni correzione che applichi**. Se in futuro scopri che
l'assunzione era sbagliata, potrai tornare indietro e capire cosa correggere
— cosa impossibile se le tue decisioni sono nascoste dentro un `dropna()`
senza commento.

## Parte 3 — Teoria: cosa fare quando un valore manca

Una volta diagnosticato (o assunto) il meccanismo, restano due decisioni
operative, e vanno prese **colonna per colonna**, non per l'intera tabella:

**Decisione A — Scartare la riga.**
Corretta quando la colonna mancante è **critica**: senza di essa la riga
perde la sua identità, cioè non sai più a cosa si riferisce. Esempio: se in
una tabella di letture manca l'orario, quella riga non è più agganciabile a
nessun momento preciso — non puoi "indovinare" quando è stata presa. Scartare
è quasi sempre l'unica opzione ragionevole per i campi identificativi (id,
timestamp, la chiave che dà senso alla riga).

**Decisione B — Imputare (riempire con un valore stimato).**
Corretta per colonne che sono **misure recuperabili**: un numero che puoi
approssimare ragionevolmente con una statistica calcolata sul resto dei
dati. Le due scelte più comuni, con effetti molto diversi:

- **media**: usa tutti i valori disponibili, pesandoli tutti allo stesso
  modo. Il problema è che un singolo valore estremo la sposta parecchio —
  te lo mostriamo tra un attimo con i numeri.
- **mediana**: il valore centrale quando ordini tutti i dati. Non si sposta
  quasi per niente se aggiungi un valore estremo, perché guarda solo *la
  posizione*, non *la distanza*. Per questo è quasi sempre la scelta più
  sicura quando non sai se i dati sono "puliti" o hanno valori anomali.

Nessuna delle due, però, **ricrea davvero la misura persa**: sta solo
mettendo un'ipotesi plausibile al posto di un buco. Questo ha due
conseguenze che è facile dimenticare:

1. Se riempi *tante* celle con lo stesso identico numero centrale, crei un
   accumulo artificiale di valori tutti uguali che nella realtà non esiste,
   e la variabilità osservata dei dati (quanto sono "sparsi") si riduce
   artificialmente. Lo vedremo con un grafico tra poco.
2. Dopo aver riempito, il valore imputato è **indistinguibile** dal valore
   originale — a meno che tu non abbia salvato *prima* dell'imputazione una
   colonna separata che dice "questo valore era mancante". Quella colonna si
   chiama comunemente un **flag di imputazione**, ed è economica da creare
   ma preziosissima: ti permette di rifare l'analisi in modo diverso più
   avanti, di dare al modello l'informazione "questo dato è stimato, non
   osservato", e di fare audit su quante celle hai davvero modificato.

In [ ]:
import pandas as pd

# Vediamo l'effetto di un valore estremo su media e mediana, con numeri veri
temperature_normali = pd.Series([18.2, 18.7, 19.1, 18.9, 18.5])
print("Solo valori normali:")
print(f"  media   = {temperature_normali.mean():.2f}")
print(f"  mediana = {temperature_normali.median():.2f}")
print("  (con dati regolari, media e mediana quasi coincidono)")

temperature_con_anomalia = pd.Series([18.2, 18.7, 19.1, 18.9, 41.0])  # un guasto del sensore
print("\nStessi dati, ma l'ultimo valore è un guasto del sensore (41.0 invece di ~18.5):")
print(f"  media   = {temperature_con_anomalia.mean():.2f}   <- spostata di quasi 5 gradi!")
print(f"  mediana = {temperature_con_anomalia.median():.2f}   <- appena mossa")

Guarda la differenza: un solo valore anomalo su cinque ha spostato la media
di quasi 5 gradi, mentre la mediana è rimasta quasi ferma. Questo è esattamente
il motivo per cui la mediana è la scelta di default quando non sei sicuro
della "pulizia" dei tuoi dati — e nei dati reali, specialmente quelli
provenienti da sensori o inseriti da persone, gli errori isolati sono la
norma, non l'eccezione.

Ora vediamo l'altro effetto collaterale annunciato: cosa succede alla
**forma della distribuzione intera** quando riempi molti buchi con lo stesso
valore centrale. Simuliamo 300 misure "vere", ne nascondiamo il 30% (come se
fossero andate perse), e confrontiamo l'istogramma prima e dopo
l'imputazione:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
misure_vere = pd.Series(rng.normal(18, 4, 300))              # la realtà, che normalmente non conosceremmo

misure_osservate = misure_vere.copy()
indici_persi = rng.choice(300, 90, replace=False)             # perdiamo il 30% a caso (scenario MCAR)
misure_osservate[indici_persi] = np.nan

misure_imputate = misure_osservate.fillna(misure_osservate.median())

fig, assi = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
assi[0].hist(misure_vere, bins=30, color="#4C72B0")
assi[0].set_title(f"Le 300 misure vere\n(varianza = {misure_vere.var():.1f})")
assi[1].hist(misure_imputate, bins=30, color="#DD8452")
assi[1].set_title(f"Dopo aver riempito il 30% mancante\ncon la mediana (varianza = {misure_imputate.var():.1f})")
plt.tight_layout()
plt.show()

Nel grafico di destra vedi una barra molto più alta esattamente al valore
della mediana: sono le 90 celle che abbiamo riempito tutte con lo stesso
identico numero. Quella barra **non esiste nella realtà** — l'abbiamo creata
noi con l'imputazione — e la varianza (quanto i dati sono "sparsi" attorno
alla media) è scesa visibilmente rispetto all'originale. I dati sembrano più
"ordinati" e prevedibili di quanto siano davvero: un rischio concreto se poi
userai queste misure per calcolare statistiche o addestrare un modello,
perché quel modello penserà di lavorare su dati meno rumorosi del vero.

Questo è il motivo per cui diciamo che **l'imputazione non è mai una
trasformazione neutra**: è un compromesso accettabile — meglio un valore
plausibile che un buco che blocca ogni calcolo — ma va sempre accompagnato
dal flag che segnala dove hai messo mano, così che tu (o chi userà questi
dati dopo di te) possa sapere cosa è osservato e cosa è stimato.

## Parte 4 — Esercizio guidato

Ora mettiamo in pratica tutta la teoria vista finora su un caso realistico:
120 letture orarie prodotte da una rete di sensori ambientali (temperatura e
umidità). Il dataset ha buchi veri, e **non ti anticipiamo dove**: la
diagnosi — guardare quante celle mancano, per colonna — è parte
dell'esercizio, esattamente come lo sarebbe nel lavoro reale.

Il tuo compito, applicando esattamente i concetti della Parte 3:

1. **misura** l'assenza per colonna (prima si misura l'entità del problema,
   solo dopo si decide cosa fare — mai il contrario);
2. **decidi i campi critici**: quali colonne, se mancano, fanno perdere
   l'identità alla riga? (Ripensa alla Parte 3, Decisione A: id, stazione e
   istante temporale identificano *quale* lettura stai guardando — senza uno
   di questi tre non sai più di cosa parli. Temperatura e umidità, invece,
   sono misure: se mancano puoi ancora dire "è la lettura della stazione X
   delle ore Y", solo con un valore da stimare.)
3. **scarta** le righe che hanno perso l'identità (Decisione A applicata);
4. **imputa** le misure recuperabili con la mediana dei dati rimasti dopo lo
   scarto (Decisione B, con la mediana perché — come visto sopra — resiste
   meglio a eventuali valori anomali), creando **prima** i flag che
   registrano dove hai riempito.

Il primo passo, la diagnosi, te lo mostro io — guarda come si misura
l'assenza per colonna con `isna()` (che restituisce `True`/`False` per ogni
cella) seguito da `mean()` (che, su una colonna di `True`/`False`, calcola
automaticamente la frazione di `True`, perché True vale 1 e False vale 0):

In [ ]:
from pathlib import Path

raw = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'environmental_sensor_missing_challenge.csv')
print(f"La tabella ha {raw.shape[0]} righe e {raw.shape[1]} colonne.")
raw.isna().mean().round(3)  # frazione di valori mancanti per colonna

Leggi il risultato come un medico legge le analisi del sangue: ogni riga
dell'output è una colonna della tabella, e il numero accanto è la percentuale
di celle vuote in quella colonna. Ora tocca a te applicare la Parte 3.

**Prova tu nella cella qui sotto**, senza guardare la soluzione. Lo schema da
seguire, passo per passo:

1. lavora su una **copia** del DataFrame (`raw.copy()`), non modificare mai
   la tabella originale mentre esplori — è buona pratica generale, non solo
   di questa lezione;
2. `dropna(subset=[...])` sulle tre colonne critiche identificate al punto 2
   sopra;
3. crea le due colonne flag (`temperature_c_was_missing` e
   `humidity_pct_was_missing`), assegnando il risultato di `.isna()` sulla
   colonna corrispondente — **prima** di riempire, altrimenti il flag
   direbbe sempre `False` perché il buco non ci sarebbe già più;
4. riempi `temperature_c` e `humidity_pct` con `fillna(mediana)`, calcolando
   la mediana **dopo** lo scarto del punto 2 (sulle righe sopravvissute, non
   su quelle originali che stai per buttare).

In [ ]:
pulito = raw.copy()

# Scrivi qui il tuo tentativo:
# 1) elimina le righe con reading_id, station_id o recorded_at mancanti
# 2) crea temperature_c_was_missing e humidity_pct_was_missing (PRIMA di imputare)
# 3) riempi temperatura e umidita' con la mediana delle righe rimaste

pulito.head()

### Soluzione spiegata

Confronta il tuo tentativo con questa versione, riga per riga:

In [ ]:
CAMPI_CRITICI = ['reading_id', 'station_id', 'recorded_at']

soluzione = raw.copy()
righe_prima = len(soluzione)

# Passo 2-3: scarto delle righe senza identità
soluzione = soluzione.dropna(subset=CAMPI_CRITICI).reset_index(drop=True)

# Passo 4: flag PRIMA di riempire, poi imputazione con la mediana dei sopravvissuti
for colonna in ['temperature_c', 'humidity_pct']:
    soluzione[f'{colonna}_was_missing'] = soluzione[colonna].isna()
    soluzione[colonna] = soluzione[colonna].fillna(soluzione[colonna].median())

report = {
    'righe_prima': righe_prima,
    'righe_dopo': len(soluzione),
    'righe_scartate': righe_prima - len(soluzione),
    'temperature_imputate': int(soluzione['temperature_c_was_missing'].sum()),
    'umidita_imputate': int(soluzione['humidity_pct_was_missing'].sum()),
}
assert soluzione[CAMPI_CRITICI].notna().all().all()  # nessun buco resta nei campi critici
assert soluzione[['temperature_c', 'humidity_pct']].notna().all().all()  # nessun buco nelle misure
report

Ripercorriamo perché ogni riga è scritta così:

- `dropna(subset=CAMPI_CRITICI)` scarta **solo** le righe che mancano di uno
  dei tre campi critici — non tutte le righe con un buco qualsiasi. Se
  scartassi ogni riga con almeno un buco (anche solo in temperatura), butteresti
  via molti dati recuperabili: è l'errore più comune di chi applica questa
  tecnica senza aver fatto la distinzione della Parte 3.
- Il flag (`isna()`) va calcolato **prima** di `fillna()`. Se invertissi
  l'ordine, quando arrivi a controllare `isna()` il buco non ci sarebbe già
  più (l'avresti già riempito), e il flag risulterebbe sempre `False` —
  perdendo per sempre l'informazione su dove hai fatto una stima.
- La mediana si calcola **dopo** il `dropna`, sulle righe sopravvissute: se
  la calcolassi prima, includeresti nel calcolo anche le righe che stai per
  buttare via, contaminando la statistica con dati che non faranno parte del
  risultato finale.
- Gli `assert` finali sono un controllo automatico: se in futuro modifichi
  questo codice e rompi accidentalmente la logica (per esempio dimenticando
  di riempire una colonna), il notebook si ferma con un errore invece di
  produrre silenziosamente un risultato sbagliato.

## Parte 5 — Il progetto: Memory AI Lab, passo 1

Da questa lezione in poi costruisci **un unico progetto** che cresce fino a
diventare un sistema completo: il **Memory AI Lab**. L'idea è un sistema che
riceve "memorie" testuali — eventi accaduti, preferenze espresse, fatti
osservati — le pulisce, le classifica, le rende ricercabili, e nelle fasi
avanzate del corso le collegherà a modelli di linguaggio.

**Il passo di oggi è l'ingestion**: la porta d'ingresso dei dati, dove si
applica esattamente la stessa teoria appena vista, ma su un dominio diverso.
Ogni record di memoria ha questo schema:

| campo | ruolo | corrisponde a, nell'esercizio sopra |
|---|---|---|
| `memory_id` | identità → **critico** | `reading_id` |
| `text` | il contenuto della memoria → **critico** | (nuovo: senza testo la memoria è vuota) |
| `timestamp` | quando è successo → **critico** | `recorded_at` |
| `type` | episodic / preference / semantic → recuperabile | (nuovo: se manca, mettiamo `"unknown"`, una costante invece di una mediana, perché è una categoria non un numero) |
| `importance` | punteggio numerico 0–1 → recuperabile | `temperature_c` / `humidity_pct` |

Nella realtà questi record arrivano da pipeline di estrazione automatica che
a volte falliscono a metà: un timeout di rete perde il testo, un parser non
trova la data. È **la stessa identica teoria di oggi**, applicata a una
tabella diversa — solo lo schema delle colonne cambia:

In [ ]:
memorie = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'memory_events_raw.csv')
print('Batch di memorie in ingresso:')
memorie

In [ ]:
CRITICI_MEMORIA = ['memory_id', 'text', 'timestamp']

progetto = memorie.dropna(subset=CRITICI_MEMORIA).reset_index(drop=True)

# 'type' è una categoria testuale: qui il riempimento naturale è una costante
# dichiarata ("unknown"), non una statistica come la mediana.
progetto['type_was_missing'] = progetto['type'].isna()
progetto['type'] = progetto['type'].fillna('unknown')

# 'importance' è numerica: stessa logica di temperatura/umidità sopra.
progetto['importance_was_missing'] = progetto['importance'].isna()
progetto['importance'] = progetto['importance'].fillna(progetto['importance'].median())

destinazione = Path('..') / 'datasets' / 'processed' / 'memory_events_clean.csv'
destinazione.parent.mkdir(parents=True, exist_ok=True)
progetto.to_csv(destinazione, index=False)

print(f'Memorie in ingresso : {len(memorie)}')
print(f'Memorie conservate  : {len(progetto)}')
print(f'Scartate (identita) : {len(memorie) - len(progetto)}')
print(f"Type imputati       : {int(progetto['type_was_missing'].sum())}")
print(f"Importance imputate : {int(progetto['importance_was_missing'].sum())}")
progetto

Il progetto ora ha il suo primo componente: un archivio di memorie pulito e
tracciabile, salvato in `datasets/processed/memory_events_clean.csv`. La
prossima lezione partirà **da questo file** e aggiungerà il secondo
componente — quindi assicurati di aver eseguito questa cella prima di
proseguire.

## Cosa hai imparato

- I dati mancanti hanno **una causa**, classificabile in tre meccanismi
  (MCAR, MAR, MNAR); la causa determina se puoi correggere il problema e
  come — questo vale per qualsiasi dataset, non solo per gli esempi di oggi.
- Un campo è **critico** quando la riga perde identità senza di esso: si
  scarta. Una misura è **recuperabile**: si imputa.
- La **mediana** resiste ai valori anomali molto meglio della media —
  verificato sopra con numeri concreti.
- L'imputazione **altera la distribuzione dei dati** (crea un accumulo
  artificiale, riduce la varianza osservata) — verificato sopra col grafico.
- Ogni imputazione va **preceduta** da un flag che registra dove hai stimato
  un valore, per non perdere quell'informazione per sempre.
- Una pulizia ben fatta produce anche un **report numerico** delle decisioni
  prese (quante righe scartate, quante celle imputate): rende il lavoro
  verificabile da chiunque, incluso te stesso tra sei mesi.

## Quiz

Prova a rispondere a mente prima di aprire le risposte — è lì che il
concetto si fissa davvero, non nel leggerlo.

1. Una stazione meteo perde soprattutto le letture durante i picchi di
   caldo estremo. A quale dei tre meccanismi di Rubin corrisponde, e perché
   scartare quelle righe distorcerebbe l'analisi?
2. Nell'esercizio guidato, perché la mediana per riempire `temperature_c`
   viene calcolata **dopo** aver scartato le righe senza campi critici, e
   non prima?
3. Se creassi il flag `temperature_c_was_missing` **dopo** aver già chiamato
   `fillna()`, cosa otterresti, e perché è sbagliato?

<details>
<summary><b>Apri le risposte</b></summary>

1. È MNAR (*Missing Not At Random*): l'assenza dipende dal valore stesso che
   manca (le temperature molto alte). Le righe che restano sottorappresentano
   proprio le giornate più calde, quindi qualunque media o modello costruito
   su di esse sottostimerà sistematicamente il fenomeno che vuoi misurare —
   esattamente il rischio descritto nella Parte 2 per il caso 3.
2. Perché le righe scartate non fanno più parte del dataset che userai: se
   la loro temperatura entrasse nel calcolo della mediana, quella statistica
   descriverebbe un insieme di dati (comprese le righe senza identità) che
   nella tabella finale non esiste più — la mediana sarebbe calcolata su
   qualcosa di diverso da quello a cui viene applicata.
3. Otterresti un flag sempre uguale a `False` per ogni riga: nel momento in
   cui controlli `isna()`, il buco è già stato riempito da `fillna()`, quindi
   non c'è più nessun valore mancante da segnalare. Perderesti per sempre
   l'informazione su quali valori erano stime e quali erano osservazioni
   vere — l'intero punto del flag, spiegato nella Parte 3.

</details>

## Fonti

- Rubin, D. B. (1976), *Inference and Missing Data*, Biometrika 63(3):
  581–592 — l'articolo originale che definisce i tre meccanismi MCAR/MAR/MNAR
  usati nella Parte 2. https://doi.org/10.1093/biomet/63.3.581
- NIST/SEMATECH, *e-Handbook of Statistical Methods*, sezione *Measures of
  Location* — la sensibilità di media e mediana ai valori estremi, mostrata
  sopra con i numeri.
  https://www.itl.nist.gov/div898/handbook/eda/section3/eda351.htm
- scikit-learn, *Imputation of missing values* — le strategie di
  imputazione (costante, media, mediana) e il ruolo dei flag/indicatori.
  https://scikit-learn.org/stable/modules/impute.html

## Prossima lezione

Arriva un **secondo batch** di memorie per il progetto — e questa volta
contiene righe duplicate, numeri scritti come testo, e punteggi
impossibili (fuori dal range che ha senso per il dominio). Servono tre nuove
difese, e le costruiamo nella Lezione 2, che riparte esattamente dal file
`memory_events_clean.csv` che hai appena salvato.